# Running the bench: one mode per section

Companion of the help page (the **?** icon of the dock, `osmo-info`, http://127.0.0.1:8080/info.html) and of
[lancer-pas-a-pas.ipynb](lancer-pas-a-pas.ipynb) (the same bench, process by process). This notebook installs
nothing: it assumes the bench of [installer-osmo-jusquau-dsp.ipynb](installer-osmo-jusquau-dsp.ipynb) or the ISO.

Each mode has three cells: **start**, **check**, **stop**. Run one mode at a time. The phone screens are shown
here, inline (`screen(n)`), and you can press keys (`keys(n, ...)`); the GTK window is `c123 --instance N`.

| Instance | What | Window |
|---|---|---|
| 1 | the phone of `--bts-classique` (osmocom or Magnetite), or the TRX of `--bts` | `c123` (`c123 --instance 1`) |
| 2 | the phone of `--bts --ms` (over the air) | `c123-1` (`c123 --instance 2`) |
| 3 | the 2nd TRX phone of `--ts 2` (TS1) | `c123-2` (`c123 --instance 3`) |
| 4 | the `rssi` app (`banc-rssi`) | `c123-3` (`c123 --instance 4`) |

The commands need root (the container runs as root; on a desktop, start Jupyter after `sudo -v`).

In [1]:
# Helpers: run a command and stream it, show a phone screen, press keys, check the network.
import os, socket, struct, subprocess, time
from IPython.display import display
from PIL import Image

OP = "/opt/GSM/osmo-operator"
ALIASES = f"{OP}/scripts/osmo-banc-aliases.sh"

def sh(cmd, timeout=900):
    """bash -c with the bench aliases (banc-rssi, c123...) loaded; output streamed."""
    p = subprocess.Popen(["bash", "-c", f". {ALIASES} 2>/dev/null; {cmd}"], cwd=OP,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0 = time.time()
    for line in p.stdout:
        print(line, end="")
        if time.time() - t0 > timeout:
            p.kill(); print("... timeout"); break
    return p.wait()

def _lcd_path(n):
    return "/dev/shm/calypso_lcd" if n == 1 else f"/dev/shm/calypso{n}/calypso_lcd"

def screen(n=1, scale=4):
    """The emulated screen of instance n: C123 (ST7558, 1 bit) or C139 (colour RGB565)."""
    b = open(_lcd_path(n), "rb").read()
    w, h, pages, flags, _, seq = struct.unpack_from("<HHHBBI", b, 8)
    d = b[32:]
    if b[:7] == b"C139LCD":
        im = Image.new("RGB", (w, h))
        px = [((p >> 11 & 31) << 3, (p >> 5 & 63) << 2, (p & 31) << 3)
              for p in struct.unpack_from("<%dH" % (w * h), d)]
        im.putdata(px)
    else:
        im = Image.new("1", (w, h), 1)
        for y in range(h):
            for x in range(w):
                if d[(y >> 3) * w + x] & (1 << (y & 7)):
                    im.putpixel((x, y), 0)
    display(im.resize((w * scale, h * scale), Image.NEAREST))

def keys(n, *names, wait=0.6):
    """Press keys on instance n: '0'..'9', 'ok', 'up', 'down', 'left', 'right', 'lsb', 'rsb', 'menu', 'power'."""
    path = "/tmp/calypso_keys.sock" if n == 1 else f"/tmp/calypso_keys{n}.sock"
    s = socket.socket(socket.AF_UNIX, socket.SOCK_DGRAM)
    for k in names:
        s.sendto(f"press {k}\n".encode(), path); time.sleep(0.25)
        s.sendto(f"release {k}\n".encode(), path); time.sleep(wait)

def vty(port, cmd):
    """One VTY command (4254 MSC, 4242 BSC, 4241 BTS, 4247 mobile)."""
    with socket.create_connection(("127.0.0.1", port), timeout=5) as c:
        c.sendall(f"enable\n{cmd}\nexit\n".encode()); time.sleep(1.0)
        out = b""
        c.settimeout(1.0)
        try:
            while chunk := c.recv(65536): out += chunk
        except socket.timeout:
            pass
    return out.decode(errors="replace")

def subscribers():
    """IMSI and cell of every subscriber attached at the MSC."""
    lines = [l.strip() for l in vty(4254, "show subscriber cache").splitlines()]
    for l in lines:
        if l.startswith(("IMSI:", "MSISDN:", "LAC / cell ID:")):
            print(l)

## 1. Classic BTS + osmocom phone (the default)

`osmo-bts-trx` talks to `pont/pont_dsp.py`; the emulated Calypso runs the osmocom-bb `layer1` firmware and `mobile`,
on the real DSP mask ROM. This is what the red phone icon starts.

In [ ]:
sh("./start.sh --regen --no-attach --bts-classique --ms")

  hub        deduit du remote-ip de l ASP : 127.0.0.1
[ .. ] Chargement de l'environnement
[ OK ] Chargement de l'environnement (environment/load.env)
[ .. ] Resolution de run.sh
[ OK ] Resolution de run.sh (/opt/GSM/qosmo/run.sh)
[ .. ] Lanceur QEMU
[ OK ] Lanceur QEMU (/usr/local/bin/qosmo (qosmo local (/opt/GSM/qosmo)))
[ .. ] Validation des chemins
[ OK ] Validation des chemins
  plan radio  lu dans /etc/osmocom/radio-plan.env
  PLMN MS    MCC 001  MNC 01  operateur 1  ->  IMSI 001010001000001 / 001010001000002
  radio      MS#1 sur ARFCN 1 (BTS principale)   MS#2 sur ARFCN 614 (side-car, unit-id 6012)
[ .. ] Generation mobile MS#1
  MS#1       aussi ecrit dans /root/.osmocom/bb/mobile_group1.cfg (le fichier que run.sh ouvre)
[ OK ] Generation mobile MS#1 (/root/.osmocom/bb/mobile.cfg)
[ .. ] Generation mobile MS#2 (faketrx)
[ OK ] Generation mobile MS#2 (faketrx) (/root/.osmocom/bb/mobile_faketrx_bts1.cfg)

╔══════════════════════════════════════════════════════╗
║   Calypso GSM -

In [ ]:
# Check: BTS#0 alive, the phone camped and registered (allow ~1 min after the start)
sh("pgrep -af 'osmo-bts-trx -c /etc/osmocom/osmo-bts-trx.cfg$' | head -1; tail -2 /dev/shm/pont.log")
subscribers()

In [3]:
sh("./start.sh --stop")

  hub        deduit du remote-ip de l ASP : 127.0.0.1
[ .. ] Chargement de l'environnement
[ OK ] Chargement de l'environnement (environment/load.env)
[ .. ] Resolution de run.sh
[ OK ] Resolution de run.sh (/opt/GSM/qosmo/run.sh)
[ .. ] Lanceur QEMU
[ OK ] Lanceur QEMU (/usr/local/bin/qosmo (qosmo local (/opt/GSM/qosmo)))
[ .. ] Validation des chemins
[ OK ] Validation des chemins
  plan radio  lu dans /etc/osmocom/radio-plan.env
  PLMN MS    MCC 001  MNC 01  operateur 1  ->  IMSI 001010001000001 / 001010001000002
  radio      MS#1 sur ARFCN 1 (BTS principale)   MS#2 sur ARFCN 614 (side-car, unit-id 6012)
[ .. ] Generation mobile MS#1
  MS#1       aussi ecrit dans /root/.osmocom/bb/mobile_group1.cfg (le fichier que run.sh ouvre)
[ OK ] Generation mobile MS#1 (/root/.osmocom/bb/mobile.cfg)
[ .. ] Generation mobile MS#2 (faketrx)
[ OK ] Generation mobile MS#2 (faketrx) (/root/.osmocom/bb/mobile_faketrx_bts1.cfg)

╔══════════════════════════════════════════════════════╗
║   Calypso GSM -

0

## 2. Classic BTS + C139 Magnetite (the real TI firmware)

Same network; the phone is a Motorola C139 running FreeCalypso Magnetite (its own L2/L3 and UI). Calls need A5/0
(with A5/1 the FACCH fails): `ENCRYPTION="a5 0"` with `--regen`. `--gapk` puts the voice on the host.
Building the firmware: `sudo ./addition.sh --magnetite` (Wine + TI TMS470).

In [5]:
sh('ENCRYPTION="a5 0" ./start.sh --regen --no-attach --bts-classique --ms-magnetite --gapk')

NameError: name 'sh' is not defined

In [6]:
# The C139 screen (colour). Its traces are in /tmp/c54x-pont/rvinterf.log.
screen(1, scale=5)
sh("grep -aE 'L1: (NP_I|RA_C|IAS_C|DDS_R)' /tmp/c54x-pont/rvinterf.log | tail -4")
subscribers()

NameError: name 'screen' is not defined

In [7]:
# Call the echo test 600 from the C139 keypad, then look at the screen
keys(1, "6", "0", "0", "ok")
time.sleep(8); screen(1, scale=5)

NameError: name 'keys' is not defined

In [8]:
sh("./start.sh --stop")

NameError: name 'sh' is not defined

## 3. The Calypso as the BTS radio (`--bts`)

The Calypso of QEMU runs the `trx` firmware and the host `transceiver`: it **is** the radio of BTS#0. A second
Calypso (`--ms`, instance 2) is the phone and camps on it over the air (`pont/air.py`).

| `--ts` | Timeslots | State |
|---|---|---|
| `--ts 1` | TS0 only (CCCH + SDCCH/4) | LU, SMS, no call |
| `--ts 2/1` | TS0 + TS1 by one Calypso (default of `--bts`) | calls |
| `--ts 2` | TS0 and TS1 by one Calypso each (instances 1 and 3) | camps, LU; no call yet (2nd TRX not frame-aligned) |

In [9]:
sh("./start.sh --regen --no-attach --bts --ms --ts 2")

NameError: name 'sh' is not defined

In [ ]:
# The TRX (instance 1), the phone over the air (instance 2), the 2nd TRX (instance 3)
for n in (2,):
    print(f"instance {n}"); screen(n)
sh("tail -3 /tmp/c54x-pont2/air.log 2>/dev/null; tail -3 /tmp/c54x-pont/transceiver.log 2>/dev/null")
subscribers()

In [ ]:
sh("./start.sh --stop")

## 4. `banc-rssi`: the rssi app on a 4th Calypso

The osmocom-bb `rssi` app (spectrum, sync, per-timeslot levels) runs on instance 4. In `--bts-classique` the pont
copies every downlink (and uplink) burst to it (UDP 6762, `PONT_DSP_COPIES`); with `--bts` it listens to the air.
The realistic air (`CALYPSO_AIR_NIVEAUX=1`) plays the cell only on its ARFCN, at -60 dBm downlink and -75 dBm uplink,
with noise elsewhere. Start a network first (section 1 or 2), then:

In [ ]:
sh("banc-rssi")

In [ ]:
# Spectrum view: ARFCN 1 (the cell) should read about -60 dBm, the next ARFCN the noise floor
time.sleep(20); screen(4)
keys(4, "right"); time.sleep(3); screen(4)     # ARFCN 2: noise
keys(4, "left")

In [ ]:
# Sync on the cell (OK), then the per-timeslot view (Up), then the uplink (right soft key)
keys(4, "ok"); time.sleep(40); screen(4)       # BSIC 7/7, MCC/MNC/LAC, cell id
keys(4, "up"); time.sleep(5); screen(4)        # 8 timeslots, downlink: all at about -60
keys(4, "rsb"); time.sleep(5); screen(4)       # uplink: the timeslot of the phone (when it transmits)

In [ ]:
sh("banc-rssi --stop")

## 5. Windows, consoles, stop

- Phone windows: `c123 --instance N` (`c123`, `c123-1`, `c123-2`, `c123-3`, `c123-tous` for all of them).
- Consoles: `sudo tmux attach -t calypso`; VTY: MSC 4254, BSC 4242, BTS 4241, mobile 4247.
- DSP bench logs: `/tmp/c54x-pont*/*.log`; `cd /opt/GSM/c54x_exe && ./run.sh --logs`.
- BTS#0 runs under a watchdog (`BTS_GARDIEN=1`): a "PC clock skew too high" exit restarts it after 3 s.
- Nothing telecom starts at boot (`services/80-osmo-operator.preset`): `systemctl enable osmo-banc` to change that.

In [ ]:
sh("./start.sh --stop; banc-rssi --stop")